# Nova-ST - Exp93-sampleSPC022

## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import glob
import os
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import gc
import spatialdata_plot
from spatialdata.transformations import get_transformation, remove_transformation, set_transformation, Identity
import dask

matplotlib.use("Agg")

import logging
logging.getLogger('ome_zarr').setLevel(logging.ERROR) # This mutes unnecessary Zarr warning (see: https://github.com/scverse/spatialdata/issues/400)

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _novast import novast

## Create sdata

In [4]:
dataset_id = "Exp93-sampleSPC022"
seq1st_id = "NovaSeq6000_20241107"
chip_id = "08A2"
path_novascope = f"/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{dataset_id}/NovaScope/novascope_output"
path_img = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{dataset_id}/BigWarp/warped.tif'
output_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{dataset_id}/harpy/sdata.zarr'

pipeline_name = 'NovaScope'
species = 'human'
tissue_annotation_file = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{dataset_id}/BigWarp/tissue_warped.geojson'

In [5]:
# sdata = sd.read_zarr(output_path)
# sdata

In [6]:
sdata = novast(
    path = path_novascope,
    dataset_id = dataset_id,
    seq1st_id = seq1st_id,
    chip_id = chip_id,
    path_img = path_img,
    microns_per_pixel= 1,
    output = output_path
)
sdata

2026-02-06 22:55:20.396 | WARNING  | harpy.io._spatial_data:_load_image_to_dask:231 - dims parameter is equal to ['c', 'y', 'x'], but will be ignored when reading in images from a file
2026-02-06 22:55:24.786 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Exp93-sampleSPC022_image'
/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
2026-02-06 23:12:42.753 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Exp93-sampleSPC022_labels'


SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/sdata.zarr
├── Images
│     └── 'Exp93-sampleSPC022_image': DataTree[cyx] (3, 6940, 11552), (3, 3470, 5776), (3, 1735, 2888), (3, 867, 1444), (3, 433, 722)
├── Labels
│     └── 'Exp93-sampleSPC022_labels': DataArray[yx] (69388, 115510)
├── Shapes
│     └── 'Exp93-sampleSPC022_shapes': GeoDataFrame shape: (22149248, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (22149248, 36601)
with coordinate systems:
    ▸ 'Exp93-sampleSPC022', with elements:
        Exp93-sampleSPC022_image (Images), Exp93-sampleSPC022_labels (Labels), Exp93-sampleSPC022_shapes (Shapes)

In [7]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

NovaScope levels: ['table']


In [8]:
# Assert whether number of shapes matches number of label IDs
import dask.array as da

for level in levels:
    number_of_shapes = len(sdata.shapes[f"{dataset_id}_shapes"])
    number_of_label_IDs = len(da.unique(sdata.labels[f"{dataset_id}_labels"].data).compute()) - 1
    assert number_of_shapes == number_of_label_IDs, f"{level}: Number of shapes ({number_of_shapes}) does not match number of label IDs ({number_of_label_IDs})"

## Calculate pixel size

In [9]:
microns_per_pixel = 1
print(f"Pixel size: {microns_per_pixel:.3f} µm")

Pixel size: 1.000 µm


## Hexagonal tesselation

In [10]:
custom_bins = ['hexagonal_grid_20um', 'hexagonal_grid_120um'] # Names refer to the diagonal of the hexagon

In [11]:
shape = (sdata.images[f'{dataset_id}_image']['scale0']['image'].shape[1], sdata.images[f'{dataset_id}_image']['scale0']['image'].shape[2])

for custom_bin in custom_bins:
    gc.collect()
    
    size = (int(custom_bin.split("hexagonal_grid_")[1].split("um")[0])/2) / microns_per_pixel # Radius of the hexagon in pixels
    
    # Create hexagonal grids
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape,
        size=size,
        output_shapes_layer=f"{custom_bin}_shapes", 
        output_labels_layer=f"{custom_bin}_labels", 
        grid_type="hexagon", 
        offset=(0, 0), 
        chunks=2048, 
        client=None, 
        transformations=get_transformation(sdata.images[f"{dataset_id}_image"], get_all=True),
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

    # Rebin data based on hexagonal grid
    sdata = hp.tb.bin_counts( 
        sdata,
        table_layer="table",
        labels_layer=f"{custom_bin}_labels",
        output_layer=f"{custom_bin}_table",
        append = False,
        to_coordinate_system = dataset_id,
        overwrite=True,
    )

2026-02-06 23:57:19.130 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'hexagonal_grid_20um_labels'
2026-02-06 23:58:15.608 | INFO     | harpy.table._allocation:_aggregate:488 - Calculating cell counts.
2026-02-06 23:59:06.941 | INFO     | harpy.table._allocation:bin_counts:363 - 
 Number of spots per bin  Frequency
                       1      11436
                       2       3518
                       3       1332
                       4        730
                       5        580
                       6        567
                       7        611
                       8        648
                       9        766
                      10        845
                      11        958
                      12       1072
                      13       1223
                      14       1316
                      15       1391
                      16       1591
                      17       1616
                      18       1749
       

## Specify plotting parameters

In [12]:
DPI = 300
MAX_DIM_FOR_PLOTS = 10000

HE_image = f'{dataset_id}_image'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale1
figsize: (19.253333333333334, 11.566666666666666)
plots_path: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots


## Expand labels layers for plotting

In [13]:
# Expand labels layer masks for plotting (otherwise, the plots are not very clear)
for level in ["table"]:
    
    radius = np.sqrt(sdata.shapes[f"{dataset_id}_shapes"].geometry.iloc[0].area / np.pi)
    
    sdata = hp.im.expand_labels_layer(
        sdata,
        labels_layer=f"{dataset_id}_labels",
        distance=(radius * 1.5) - radius, # Radius increased by 150%
        output_labels_layer=f"{dataset_id}_labels",
        overwrite=True,
)
    
# NOTE: The result of this is that plots using the labels layer do not show spots to scale, but the visualization is improved.

2026-02-07 00:06:32.763 | INFO     | harpy.image.segmentation._map:_combine_dask_arrays:339 - Linking labels across chunks.
2026-02-07 00:06:40.438 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Exp93-sampleSPC022_labels'
2026-02-07 00:11:37.357 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'Exp93-sampleSPC022_labels' already exists. Overwriting...


## Create crop for zoomed-in plots

In [14]:
min_x = sdata.images[HE_image]['scale0']['image'].shape[2] / 2 # Center of image
max_x = min_x + round(200 / microns_per_pixel) # 200 micrometer
min_y = sdata.images[HE_image]['scale0']['image'].shape[1] / 2 # Center of image
max_y = min_y + round(200 / microns_per_pixel) # 200 micrometer

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y], 
    max_coordinate=[max_x, max_y], 
    axes=("x", "y"), 
    target_coordinate_system=dataset_id
)

figsize_crop = (3,3)

## Plot H&E

In [15]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/HE.png


## Plot NovaScope spots

In [16]:
for level in levels:
    gc.collect()

    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
            element=f"{dataset_id}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"{pipeline_name} - spots"

    sdata_crop.pl.render_images(
        element=HE_image, scale="full"
    ).pl.render_shapes(
            element=f"{dataset_id}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize_crop,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Spots are represented true to scale.

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/table/NovaScope_spots.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/table/crop_NovaScope_spots.png


## Plot hexagonal spots

In [17]:
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata_crop.pl.render_images(
        element=HE_image, scale="full"
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Spots are represented true to scale

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/hexagonal_grid_20um.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/crop_hexagonal_grid_20um.png
INFO     Value for parameter 'color' 

## Create bounding box for total capture area

In [18]:
# Add bounding box for total capture area
from shapely.affinity import affine_transform
from shapely.geometry import MultiPolygon

transformations = get_transformation(sdata.shapes[f"{dataset_id}_shapes"], get_all=True)
transformations = next(iter(transformations.values()))
affine_matrix = transformations.to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))

shapely_affine = [
    affine_matrix[0, 0],  
    affine_matrix[0, 1],  
    affine_matrix[1, 0],  
    affine_matrix[1, 1],
    affine_matrix[0, 2],
    affine_matrix[1, 2],
]

geoms = sdata.shapes[f"{dataset_id}_shapes"].geometry
bbox_transformed = MultiPolygon(list(geoms)).minimum_rotated_rectangle
bbox = affine_transform(bbox_transformed, shapely_affine)

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.images[f"{dataset_id}_image"], get_all=True),
    overwrite = True,
)

In [19]:
# Plot capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.05)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/capture_area_bbox.png


In [20]:
# Plot capture area bbox
for level in ['table']:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - Capture area bounding box"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
        element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
    ).pl.render_shapes(
        element=f'{dataset_id}_shapes', fill_alpha=0, outline_width=1, outline_alpha=1, outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'capture_area_bbox.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/table/capture_area_bbox.png


In [21]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

Area of capture area: 80.15 mm²


In [22]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

Dimensions of capture area bbox: 1.16 cm x 0.69 cm


## Add tissue annotation

In [23]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.images[f'{dataset_id}_image'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2026-02-07 00:44:56.514 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'Exp93-sampleSPC022_tissue_mask'


In [24]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'Qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/Qupath_tissue_annotation.png


In [25]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = level, 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{custom_bin}_table", 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

2026-02-07 00:48:09.178 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'table' already exists. Overwriting...
2026-02-07 00:49:39.283 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table' already exists. Overwriting...
2026-02-07 00:50:35.132 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table' already exists. Overwriting...


In [27]:
# Check spots in tissue
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{custom_bin} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{custom_bin}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=f"{custom_bin}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Qupath_tissue_annotation_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_120um/Qupath_tissue_annotation_spots.png


## QC

In [28]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = level,
        species = species
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{custom_bin}_table",
        species = species
    )

2026-02-07 00:56:43.308 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'table' already exists. Overwriting...
2026-02-07 00:58:29.982 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table' already exists. Overwriting...
2026-02-07 00:59:26.680 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table' already exists. Overwriting...


In [29]:
# Plot QC metrics
for custom_bin in custom_bins:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, custom_bin)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{custom_bin}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{custom_bin}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{custom_bin}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)
    
# QC metrics of 'table' (containing highest resolution spots) are not plotted since the plots are not very informative due to the small size of the spots.

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Scanpy_QC_n_genes_by_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts_in_top_50_genes.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts_mt.png
Plot saved at: /d

## Leiden clustering

In [30]:
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY

for custom_bin in custom_bins:
    gc.collect()
    
    print(f'running: {custom_bin}', flush=True)

    labels_layer=f"{custom_bin}_labels"
    shapes_layer=f"{custom_bin}_shapes"
    table_layer=f"{custom_bin}_table"
    
    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)
    
    # Only keep spots in tissue for clustering
    adata = sdata.tables[table_layer][sdata.tables[table_layer].obs['in_tissue_annotation'] == True]
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=f"{table_layer}_processed",
        region=adata.obs[_REGION_KEY].cat.categories.to_list(),
        overwrite=True,
    )
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False,
        labels_layer=labels_layer,
        table_layer=f"{table_layer}_processed",
        output_layer=f"{table_layer}_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps=50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.75, 1, 1.25]
    calculate_umap = True
    calculate_neighbors = True

    for res in leiden_resolutions:
        gc.collect()

        res_str = str(res).replace('.', '')
        print(f'Leiden resolution: {res}', flush=True)
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_processed",
            output_layer=f"{table_layer}_processed",
            calculate_umap=calculate_umap,
            calculate_neighbors=calculate_neighbors,
            rank_genes=False,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        calculate_umap = False # Only calculate UMAP once
        calculate_neighbors = False # Only calculate neighbors once

        # Plot Leiden clustering
        gc.collect()
        
        with matplotlib.rc_context({'legend.markerscale': 2}):
            fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

            title = f"Leiden {res}"

            sdata.pl.render_images(
                element=HE_image, scale=best_scale
            ).pl.render_labels(
                element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, cmap="rainbow", color=f'leiden_{res_str}', table_name=f"{table_layer}_processed", scale=best_scale
            ).pl.show(
                coordinate_systems=dataset_id,
                title=title,
                figsize=figsize,
                colorbar=False,
                legend_fontsize=rel_fontsize(fig, 0.015), 
                legend_fontweight='bold', 
                legend_loc='right margin',
                na_in_legend=False,
                dpi=DPI,
                ax=ax,
            )

            ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
            ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
            ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
            ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

            save_path = os.path.join(level_plots_path, f'Leiden_{res}.png')
            fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

            plt.close(fig)

            print("Plot saved at:", save_path, flush=True)
        
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(level_plots_path, f'Leiden_{res}')
        )

running: hexagonal_grid_20um


2026-02-07 01:04:24.405 | INFO     | harpy.table._preprocess:preprocess:314 - Calculating cell size from provided labels_layer 'hexagonal_grid_20um_labels'
2026-02-07 01:04:25.502 | INFO     | harpy.table._preprocess:preprocess:334 - Saving non preprocessed data matrix to the AnnData '.layers[raw_counts]'.
2026-02-07 01:05:20.822 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Leiden resolution: 0.75


2026-02-07 01:21:42.596 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_20um/Leiden_0.75.png
Leiden resolution: 1


...

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_120um/Leiden_0.75.png
Leiden resolution: 1


2026-02-07 02:03:00.363 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_120um/Leiden_1.png
Leiden resolution: 1.25


2026-02-07 02:04:48.081 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/harpy/plots/hexagonal_grid_120um/Leiden_1.25.png


## Spots metrics

In [31]:
gc.collect()

10978

In [32]:
sbcds_manifest = pd.read_csv(os.path.join(path_novascope, f'seq1st/{seq1st_id}/sbcds/{seq1st_id}/manifest.tsv'), delimiter='\t')
sbcds_manifest
# NOTE: sbcds contains the information for ALL chips that were cut from this flowcell (not just the one used for this sample).

,id,filepath,barcodes,matches,mismatches,xmin,xmax,ymin,ymax
0,1_1101,1_1101.sbcds.sorted.tsv.gz,3441661,3377626,64035,1136,32931,1016,37059
1,1_1102,1_1102.sbcds.sorted.tsv.gz,3439440,3377026,62414,1136,32931,1016,37059
2,1_1103,1_1103.sbcds.sorted.tsv.gz,3440741,3377136,63605,1127,32922,1016,37059
3,1_1104,1_1104.sbcds.sorted.tsv.gz,3441762,3378802,62960,1136,32931,1016,37059
4,1_1105,1_1105.sbcds.sorted.tsv.gz,3441644,3375816,65828,1118,32922,1016,37059
...,...,...,...,...,...,...,...,...,...
3738,4_2674,4_2674.sbcds.sorted.tsv.gz,3423140,3357938,65202,1072,32859,1031,37059
3739,4_2675,4_2675.sbcds.sorted.tsv.gz,3414367,3347807,66560,1063,32859,1031,37059
3740,4_2676,4_2676.sbcds.sorted.tsv.gz,3414774,3355234,59540,1054,32850,1047,37059
3741,4_2677,4_2677.sbcds.sorted.tsv.gz,3419366,3355813,63553,1054,32841,1047,37059


In [33]:
print('Number of tiles:', sbcds_manifest['id'].nunique())
print(f"Sum of all spatial barcodes: {sbcds_manifest['barcodes'].sum():,}")
print(f"Sum of all spatial barcodes (matches): {sbcds_manifest['matches'].sum():,}")
print(f"Sum of all spatial barcodes (mismatches): {sbcds_manifest['mismatches'].sum():,}") 

Number of tiles: 3743
Sum of all spatial barcodes: 12,827,831,696
Sum of all spatial barcodes (matches): 12,588,709,400
Sum of all spatial barcodes (mismatches): 239,122,296


In [34]:
nbcds_manifest = pd.read_csv(os.path.join(path_novascope, f'seq1st/{seq1st_id}/nbcds/{chip_id}/manifest.tsv'), delimiter='\t')
nbcds_manifest
# NOTE: nbcds contains only the information for the chip of this particular sample.

,id,filepath,barcodes,matches,mismatches,xmin,xmax,ymin,ymax
0,1_1,1_1.sbcds.sorted.tsv.gz,148952530,148952530,0,0,11550681,0,6938853


In [35]:
print(f"All spatial barcodes: {nbcds_manifest['barcodes'][0]:,}")
print(f"All spatial barcodes (matches): {nbcds_manifest['matches'][0]:,}")
print(f"All spatial barcodes (mismatches): {nbcds_manifest['mismatches'][0]:,}")
print(f"Coordinates (xmin, xmax, ymin, ymax): ({nbcds_manifest['xmin'][0]}, {nbcds_manifest['xmax'][0]}, {nbcds_manifest['ymin'][0]}, {nbcds_manifest['ymax'][0]})")

All spatial barcodes: 148,952,530
All spatial barcodes (matches): 148,952,530
All spatial barcodes (mismatches): 0
Coordinates (xmin, xmax, ymin, ymax): (0, 11550681, 0, 6938853)


In [36]:
nbcds_sorted = pd.read_csv(os.path.join(path_novascope, f'seq1st/{seq1st_id}/nbcds/{chip_id}/1_1.sbcds.sorted.tsv.gz'), delimiter='\t', header=None, names=['barcode', 'lane_id', 'tile_id', 'x', 'y', 'count'])
nbcds_sorted
# This file maps spatial barcodes to global chip coordinates, with duplicate barcodes removed.

,barcode,lane_id,tile_id,x,y,count
0,AAAAAAAAAAAAAAAAAAAAAAAAGCGAGGCG,1,1,6310622,134876,1
1,AAAAAAAAAAAAAAAAAAAAAAAGCGAGACTC,1,1,8227661,1451543,1
2,AAAAAAAAAAAAAAAAACAACGGCCGGTATTC,1,1,1787309,6618425,1
3,AAAAAAAAAAAAAAAAAGCGAGAGTCATTTTA,1,1,7264776,5677869,1
4,AAAAAAAAAAAAAACGACCGGTGAACGTTCGT,1,1,10494795,5846204,1
...,...,...,...,...,...,...
148952525,TTTTTTTTTTTTTTTTTTTTTTTTGGCCCCCA,1,1,5490544,2260560,8
148952526,TTTTTTTTTTTTTTTTTTTTTTTTGGGCCCCA,1,1,4413303,1078806,8
148952527,TTTTTTTTTTTTTTTTTTTTTTTTTAGTATCG,1,1,10212138,1960896,8
148952528,TTTTTTTTTTTTTTTTTTTTTTTTTGGCCCCT,1,1,9631590,2439608,8


In [37]:
print(f"Number of unique spatial barcodes in capture area: {nbcds_sorted['barcode'].nunique():,}")
print(f"Length of spatial barcode: {len(nbcds_sorted['barcode'][0])} bp")
print(f"Number of lanes: {nbcds_sorted['lane_id'].nunique()}") # This probably does not accurately reflect the number of lanes
print(f"Number of tiles: {nbcds_sorted['tile_id'].nunique()}") # This clearly does not accurately reflect the number of tiles
print(f"Coordinates (xmin, xmax, ymin, ymax): ({nbcds_sorted['x'].min()}, {nbcds_sorted['x'].max()}, {nbcds_sorted['y'].min()}, {nbcds_sorted['y'].max()})")
print(f"Number of unique XY coordinates: {nbcds_sorted[['x', 'y']].drop_duplicates().shape[0]:,}")

Number of unique spatial barcodes in capture area: 148,952,530
Length of spatial barcode: 32 bp
Number of lanes: 1
Number of tiles: 1
Coordinates (xmin, xmax, ymin, ymax): (0, 11550681, 0, 6938853)
Number of unique XY coordinates: 148,952,530


In [38]:
# Total number of spots in tissue
tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
mask_shape = tissue_mask.shape

nbcds_sorted["in_tissue_annotation"] = nbcds_sorted.apply(
    lambda row: assign_region((row["x"]/1000, row["y"]/1000), tissue_mask, mask_shape), axis=1
)

total_n_spots_in_tissue = len(nbcds_sorted[nbcds_sorted['in_tissue_annotation'] == 1])

print(f"Total number of spots in tissue: {total_n_spots_in_tissue:,}")

Total number of spots in tissue: 88,291,685


In [39]:
# Total number of spots in table
n_spots = len(sdata.tables['table'].obs)
print(f"Number of spots in table: {n_spots:,}")

Number of spots in table: 22,149,248


In [40]:
# Number of spots in tissue (in table)
n_spots_in_tissue = len(sdata.tables['table'][sdata.tables['table'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue:,}")

Number of spots in tissue: 18,334,560


In [41]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in nbcds_sorted['barcode']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in nbcds_sorted[nbcds_sorted['in_tissue_annotation'] == 1]['barcode']:
        f.write(f"{item}\n")      

In [42]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 47.07 mm²


## Capture effeciency metrics

In [43]:
gc.collect()

99

In [44]:
def row_sums(X):
    """Return 1D array of row sums without densifying."""
    return np.asarray(X.sum(axis=1)).ravel()

def col_sums(X):
    """Return 1D array of column sums without densifying."""
    return np.asarray(X.sum(axis=0)).ravel()

In [45]:
# Define Anndata
adata = sdata.tables["table"]

In [46]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = (row_sums(adata.X) > 0).sum()
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero:,}")

Number of spots in table with at least one UMI: 19,353,667


In [47]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = adata.obs['in_tissue_annotation'] == 1
umis_mask = row_sums(adata.X) > 0
n_spots_in_tissue_not_zero = (in_tissue_mask & umis_mask).sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero:,}")

Number of spots in tissue with at least one UMI: 16,675,078


In [48]:
# Total number of genes in table
print(f"Total number of genes in table: {(col_sums(adata.X) > 0).sum():,}")

Total number of genes in table: 24,701


In [49]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(col_sums(adata[adata.obs['in_tissue_annotation'] == 1].X) > 0).sum():,}")

Number of genes in tissue: 24,241


In [50]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum():,.0f}")

Total number of unique UMIs: 26,850,566


In [51]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum():,.0f}")

Number of unique UMIs in tissue: 23,722,915


## Additional exploration of NovaScope output

In [52]:
sum_reads = 0
sum_mismatches = 0
sum_matches = 0

for summary in glob.glob(os.path.join(path_novascope, f"match/{seq1st_id}/{chip_id}/**/*.summary.tsv"), recursive=True):
    df = pd.read_csv(summary, delimiter='\t', index_col='Type')
    sum_reads += df.at['Total', 'Reads']
    sum_mismatches += df.at['Miss', 'Reads']
    sum_matches += df.at['Match', 'Reads']

print(f"Sum of all reads: {sum_reads:,}")
print(f"Sum of all mismatches: {sum_mismatches:,} ({(sum_mismatches/sum_reads)*100:.1f} %)")
print(f"Sum of all matches: {sum_matches:,} ({(sum_matches/sum_reads)*100:.1f} %)") # These reads are used for the `align` step

# Total: All reads in the 2nd-seq FASTQ file.
# Miss: Reads that do not contain matching spatial barcodes.
# Match: Reads that match with a spatial barcode.
# Unique: Unique spatial barcodes that has matches.
# Dup(Exact): Duplicate barcodes calculated as Match - Unique.

Sum of all reads: 431,485,173
Sum of all mismatches: 215,260,843 (49.9 %)
Sum of all matches: 216,224,330 (50.1 %)


In [53]:
unique = set()
unique_xy = set()
sum_count = 0
sum_mismatches = 0

for path in glob.glob(os.path.join(path_novascope, f"match/{seq1st_id}/{chip_id}/**/*.match.sorted.uniq.tsv.gz")):
    for chunk in pd.read_csv(path, sep="\t", header=None, names=['barcode', 'lane_id', 'tile_id', 'x', 'y', 'mismatches', 'count'], usecols=[0, 3, 4, 5, 6], chunksize=500000):
        unique.update(chunk['barcode'])
        unique_xy.update(zip(chunk['x'], chunk['y']))
        sum_count += chunk['count'].sum()
        sum_mismatches += chunk['mismatches'].sum()

print(f"Number of unique spatial barcodes: {len(unique):,}")
print(f"Number of unique XY coordinates: {len(unique_xy):,}")
print(f"Sum of spatial barcode counts (i.e. number of reads containing a valid spatial barcode): {sum_count:,}") # These reads are used for the `align` step
print(f"Sum of mismatches across unique spatial barcodes: {sum_mismatches:,}")

Number of unique spatial barcodes: 66,648,355
Number of unique XY coordinates: 66,648,355
Sum of spatial barcode counts (i.e. number of reads containing a valid spatial barcode): 216,224,330
Sum of mismatches across unique spatial barcodes: 366,518


In [54]:
# STARsolo barcode statistics
bc_stats = {}
with open(os.path.join(path_novascope, f"align/{seq1st_id}/{chip_id}/{dataset_id}/bam/sttoolsSolo.out/Barcodes.stats")) as f:
    for line in f:
        line = line.strip().split()
        bc_stats[line[0]] = int(line[1])
        
total = sum(bc_stats.values())

print(f"noNoAdapter: {bc_stats['noNoAdapter']:,} ({bc_stats['noNoAdapter'] / total * 100:.1f}%)",
    " -  Number of reads without an adapter")
print(f"noNoUMI: {bc_stats['noNoUMI']:,} ({bc_stats['noNoUMI'] / total * 100:.1f}%)",
    " -  Number of reads not associated with a valid UMI")
print(f"noNoCB: {bc_stats['noNoCB']:,} ({bc_stats['noNoCB'] / total * 100:.1f}%)",
    " -  Number of reads not associated with a valid spatial barcode")
print(f"noNinCB: {bc_stats['noNinCB']:,} ({bc_stats['noNinCB'] / total * 100:.1f}%)",
    " -  Number of reads with too many Ns in the spatial barcode")
print(f"noNinUMI: {bc_stats['noNinUMI']:,} ({bc_stats['noNinUMI'] / total * 100:.1f}%)",
    " -  Number of reads with too many Ns in the UMI")
print(f"noUMIhomopolymer: {bc_stats['noUMIhomopolymer']:,} ({bc_stats['noUMIhomopolymer'] / total * 100:.1f}%)",
    " -  Number of reads that have a homopolymeric UMI")
print(f"noNoWLmatch: {bc_stats['noNoWLmatch']:,} ({bc_stats['noNoWLmatch'] / total * 100:.1f}%)",
    " -  Number of reads that do not have a match to the whitelist (even when allowing one mismatch)")
print(f"noTooManyMM: {bc_stats['noTooManyMM']:,} ({bc_stats['noTooManyMM'] / total * 100:.1f}%)",
    " -  Number of reads that have too many mismatches to the whitelist")
print(f"noTooManyWLmatches: {bc_stats['noTooManyWLmatches']:,} ({bc_stats['noTooManyWLmatches'] / total * 100:.1f}%)",
    " -  Number of reads that have too many matches to the whitelist (not used at the moment)")
print(f"yesWLmatchExact: {bc_stats['yesWLmatchExact']:,} ({bc_stats['yesWLmatchExact'] / total * 100:.1f}%)",
    " -  Number of reads with an exact match to the whitelist")
print(f"yesOneWLmatchWithMM: {bc_stats['yesOneWLmatchWithMM']:,} ({bc_stats['yesOneWLmatchWithMM'] / total * 100:.1f}%)",
    " -  Number of reads with one match to the whitelist but there are mismatches")
print(f"yesMultWLmatchWithMM: {bc_stats['yesMultWLmatchWithMM']:,} ({bc_stats['yesMultWLmatchWithMM'] / total * 100:.1f}%)",
    " -  Number of reads with multiple matches to the whitelist but there are mismatches")

print(f"\nSum of all bc_stats: {total:,} (100.00%)")

yes_sum = (
    bc_stats['yesWLmatchExact']
    + bc_stats['yesOneWLmatchWithMM']
    + bc_stats['yesMultWLmatchWithMM']
)

print(f"Sum of `yes` bc_stats: {yes_sum:,} ({yes_sum / total * 100:.2f}%)")

# For explanation about stats see: https://github.com/alexdobin/STAR/issues/1887
# prefix 'no' means that reads are not used for quantification

noNoAdapter: 0 (0.0%)  -  Number of reads without an adapter
noNoUMI: 0 (0.0%)  -  Number of reads not associated with a valid UMI
noNoCB: 0 (0.0%)  -  Number of reads not associated with a valid spatial barcode
noNinCB: 1 (0.0%)  -  Number of reads with too many Ns in the spatial barcode
noNinUMI: 389,962 (0.2%)  -  Number of reads with too many Ns in the UMI
noUMIhomopolymer: 481,607 (0.2%)  -  Number of reads that have a homopolymeric UMI
noNoWLmatch: 698,060 (0.3%)  -  Number of reads that do not have a match to the whitelist (even when allowing one mismatch)
noTooManyMM: 0 (0.0%)  -  Number of reads that have too many mismatches to the whitelist
noTooManyWLmatches: 0 (0.0%)  -  Number of reads that have too many matches to the whitelist (not used at the moment)
yesWLmatchExact: 211,070,925 (97.6%)  -  Number of reads with an exact match to the whitelist
yesOneWLmatchWithMM: 2,971 (0.0%)  -  Number of reads with one match to the whitelist but there are mismatches
yesMultWLmatchWith

In [55]:
# STARsolo feature statistics (Gene)
feature_stats = {}
with open(os.path.join(path_novascope, f"align/{seq1st_id}/{chip_id}/{dataset_id}/bam/sttoolsSolo.out/Gene/Features.stats")) as f:
    for line in f:
        line = line.strip().split()
        feature_stats[line[0]] = int(line[1])
    
print(f"noUnmapped: {feature_stats['noUnmapped']:,} ({feature_stats['noUnmapped'] / yes_sum * 100:.1f}%)  -  Number of reads that do not map to the reference genome")
print(f"noNoFeature: {feature_stats['noNoFeature']:,} ({feature_stats['noNoFeature'] / yes_sum * 100:.1f}%)  -  Number of reads that map to the genome but not to any feature")
print(f"MultiFeature: {feature_stats['MultiFeature']:,} ({feature_stats['MultiFeature'] / yes_sum * 100:.1f}%)  -  Number of reads that map to multiple features")
print(f"    - subMultiFeatureMultiGenomic: {feature_stats['subMultiFeatureMultiGenomic']:,} ({feature_stats['subMultiFeatureMultiGenomic'] / yes_sum * 100:.1f}%)  -  Reads mapping to multiple genomic loci and multiple features (a subset of MultiFeature)")
print(f"noTooManyWLmatches: {feature_stats['noTooManyWLmatches']:,} ({feature_stats['noTooManyWLmatches'] / yes_sum * 100:.1f}%)  -  Number of reads rejected because the spatial barcode has too many matches to the whitelist")
print(f"noMMtoWLwithoutExact: {feature_stats['noMMtoWLwithoutExact']:,} ({feature_stats['noMMtoWLwithoutExact'] / yes_sum * 100:.1f}%)  -  Number of reads rejected due to mismatches without an exact-match backup")
print(f"yesWLmatch: {feature_stats['yesWLmatch']:,} ({feature_stats['yesWLmatch'] / yes_sum * 100:.1f}%)  -  Reads mapping to a feature with a whitelist-matching barcode")
print(f"    - yessubWLmatchExact: {feature_stats['yessubWLmatchExact']:,} ({feature_stats['yessubWLmatchExact'] / yes_sum * 100:.1f}%)  -  Exact whitelist matches (subset of yesWLmatch)")
print(f"    - yessubWLmatch_UniqueFeature: {feature_stats['yessubWLmatch_UniqueFeature']:,} ({feature_stats['yessubWLmatch_UniqueFeature'] / yes_sum * 100:.1f}%)  -  Reads mapping uniquely to a feature (subset of yesWLmatch)")

print(f"\nyesCellBarcodes: {feature_stats['yesCellBarcodes']:,}  -  Number of unique valid spatial barcodes (i.e. Number of spots)")
print(f"yesUMIs: {feature_stats['yesUMIs']:,}  -  Number of unique valid UMIs")

mapped_sum = (
    feature_stats['yesWLmatch']
    + feature_stats['noNoFeature']
    + feature_stats['noUnmapped']
    + feature_stats['MultiFeature']
    + feature_stats['noTooManyWLmatches']
    + feature_stats['noMMtoWLwithoutExact']
)

print(f"\nSum of all mapped, unmapped and multi-mapped reads: {mapped_sum:,} ({mapped_sum / yes_sum * 100:.1f}%)")

# For explanation about stats see: https://github.com/alexdobin/STAR/issues/1887, https://github.com/alexdobin/STAR/blob/master/doc/STARmanual.pdf and https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-preprocessing-tenx/tutorial.html. 
# The explanations above are my best guesses based on the available information.
# NOTE: In addition to the gene counts (Gene; exons), STARsolo also calculates counts for the following features:
#   - pre-mRNA counts (GeneFull) (exons + introns)
#   - annotated and novel splice junctions (SJ)
#   - Velocyto spliced/unspliced/ambiguous counts (Velocyto)

noUnmapped: 114,286,680 (53.2%)  -  Number of reads that do not map to the reference genome
noNoFeature: 34,908,187 (16.3%)  -  Number of reads that map to the genome but not to any feature
MultiFeature: 1,605,930 (0.7%)  -  Number of reads that map to multiple features
    - subMultiFeatureMultiGenomic: 1,260,926 (0.6%)  -  Reads mapping to multiple genomic loci and multiple features (a subset of MultiFeature)
noTooManyWLmatches: 715,274 (0.3%)  -  Number of reads rejected because the spatial barcode has too many matches to the whitelist
noMMtoWLwithoutExact: 43 (0.0%)  -  Number of reads rejected due to mismatches without an exact-match backup
yesWLmatch: 63,138,586 (29.4%)  -  Reads mapping to a feature with a whitelist-matching barcode
    - yessubWLmatchExact: 63,044,954 (29.4%)  -  Exact whitelist matches (subset of yesWLmatch)
    - yessubWLmatch_UniqueFeature: 63,138,586 (29.4%)  -  Reads mapping uniquely to a feature (subset of yesWLmatch)

yesCellBarcodes: 19,354,549  -  Numb

In [56]:
# Summary statistics (Gene)
summary = pd.read_csv(os.path.join(path_novascope, f"align/{seq1st_id}/{chip_id}/{dataset_id}/bam/sttoolsSolo.out/Gene/Summary.csv"), sep=",", index_col=0, header=None)
print(f"Number of reads used for STARsolo: {int(summary.loc['Number of Reads'].values[0]):,}")
print(f"Sequencing saturation: {float(summary.loc['Sequencing Saturation'].values[0])*100:.1f} %")
print(f"Q30 bases in spatial barcode + UMI: {float(summary.loc['Q30 Bases in CB+UMI'].values[0])*100:.1f} %")
print(f"Q30 bases in RNA read: {float(summary.loc['Q30 Bases in RNA read'].values[0])*100:.1f} %")

Number of reads used for STARsolo: 216,224,330
Sequencing saturation: 57.5 %
Q30 bases in spatial barcode + UMI: 95.0 %
Q30 bases in RNA read: 92.7 %


In [57]:
# Additional AnnData metrics
print(f"Total counts (UMIs are deduplicated): {adata.X.sum():,}")
print(f"Sum of Gene counts: {adata.obs['Gene'].sum():,}")
print(f"Sum of GeneFull counts: {adata.obs['GeneFull'].sum():,}")
print(f"Sum of Spliced counts: {adata.obs['Spliced'].sum():,}")
print(f"Sum of Unspliced counts: {adata.obs['Unspliced'].sum():,}")
print(f"Sum of Ambiguous counts: {adata.obs['Ambiguous'].sum():,}")

Total counts (UMIs are deduplicated): 26,850,566
Sum of Gene counts: 26,850,566
Sum of GeneFull counts: 30,149,134
Sum of Spliced counts: 21,355,101
Sum of Unspliced counts: 3,418,716
Sum of Ambiguous counts: 3,411,845


In [58]:
print(f"Sequencing saturation: {(1- (feature_stats['yesUMIs']/feature_stats['yesWLmatch']))*100:,.1f} %")
print(f"Sequencing saturation: {(1- (adata.obs['Gene'].sum()/feature_stats['yesWLmatch']))*100:,.1f} %")

Sequencing saturation: 57.5 %
Sequencing saturation: 57.5 %


In [59]:
path_transcripts = os.path.join(path_novascope,f"analysis/{dataset_id}/{dataset_id}-default/preprocess/{dataset_id}-default.transcripts.tsv.gz")

transcripts = pd.read_csv(path_transcripts, delimiter='\t') # Does not contain spatial barcodes
transcripts

,#lane,tile,X,Y,gene_id,gene,gn,gt,spl,unspl,ambig
0,0,0,0,0,ENSG00000003402,CFLAR,1,1,0,0,1
1,0,0,0,0,ENSG00000004478,FKBP4,1,1,0,0,1
2,0,0,0,0,ENSG00000004809,SLC22A16,0,1,0,1,0
3,0,0,0,0,ENSG00000005893,LAMP2,1,1,0,0,1
4,0,0,0,0,ENSG00000006715,VPS41,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
30188104,1,1,11549568,5624412,ENSG00000206503,HLA-A,1,1,1,0,0
30188105,1,1,11550124,1079745,ENSG00000241772,AC092620.1,1,1,1,0,0
30188106,1,1,11550124,3234470,ENSG00000277734,TRAC,1,1,1,0,0
30188107,1,1,11550124,5701450,ENSG00000120549,KIAA1217,0,1,0,1,0


In [60]:
print(f"Coordinates (xmin, xmax, ymin, ymax): ({transcripts['X'].min()}, {transcripts['X'].max()}, {transcripts['Y'].min()}, {transcripts['Y'].max()})")
print(f"Number of unique XY coordinates: {transcripts[['X', 'Y']].drop_duplicates().shape[0]:,}")
print(f"Sum of gn: {transcripts['gn'].sum():,}")
print(f"Sum of gt: {transcripts['gt'].sum():,}")
print(f"Sum of spl: {transcripts['spl'].sum():,}")
print(f"Sum of unspl: {transcripts['unspl'].sum():,}")
print(f"Sum of ambig: {transcripts['ambig'].sum():,}")

Coordinates (xmin, xmax, ymin, ymax): (0, 11550681, 0, 6938540)
Number of unique XY coordinates: 22,149,249
Sum of gn: 26,851,839
Sum of gt: 30,150,582
Sum of spl: 21,356,110
Sum of unspl: 3,418,884
Sum of ambig: 3,412,002
